# Lab 08-03 · Logic Minimization

Companion to chapter [08-03](../../chapters/08-03-logic-minimization.md). Handout: [README.md](README.md).

A term is written as a string over `0`, `1` and `-`, one character per variable (`-1-1` is $BD$ in four variables). Everything here is built from that representation: maps, Quine–McCluskey, Petrick's method, and an exact minimizer used as the referee. Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import functools
import itertools
import time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

VARS = "ABCDEF"


def to_bits(i, n):
    return format(i, f"0{n}b")


def term_str(t):
    out = "".join(VARS[k] + ("" if ch == "1" else "'") for k, ch in enumerate(t) if ch != "-")
    return out or "1"


def covers(t, i, n):
    b = to_bits(i, n)
    return all(tc == "-" or tc == bc for tc, bc in zip(t, b))


def literals(t):
    return sum(ch != "-" for ch in t)

## Part 1 · Karnaugh maps

Rows and columns follow Gray code (00, 01, 11, 10), so neighbors differ in one bit, including across the edges.

**🔮 Predict:** which cells are adjacent to $m_0$ through the top–bottom wrap and through the left–right wrap?

In [2]:
GRAY = ["00", "01", "11", "10"]


def kmap(on, dc=(), title=""):
    fig, ax = plt.subplots(figsize=(3.6, 3.4))
    for r, ab in enumerate(GRAY):
        for c, cd in enumerate(GRAY):
            m = int(ab + cd, 2)
            val = "1" if m in on else "X" if m in dc else "0"
            ax.add_patch(plt.Rectangle((c, 3 - r), 1, 1, fill=val != "0",
                                       color="#56B4E9" if val == "1" else "#DDDDDD", alpha=0.6, ec="k"))
            ax.text(c + 0.5, 3 - r + 0.55, val, ha="center", va="center", fontsize=13)
            ax.text(c + 0.88, 3 - r + 0.15, str(m), ha="right", va="bottom", fontsize=7, color="0.35")
    ax.set_xticks([0.5, 1.5, 2.5, 3.5]); ax.set_xticklabels(GRAY)
    ax.set_yticks([3.5, 2.5, 1.5, 0.5]); ax.set_yticklabels(GRAY)
    ax.set_xlabel("CD"); ax.set_ylabel("AB"); ax.set_xlim(0, 4); ax.set_ylim(0, 4)
    ax.set_title(title); ax.set_aspect("equal")
    plt.show()


kmap({0, 2, 5, 7, 8, 10, 13, 15}, title="Example 08-03.2: BD + B'D'")
kmap({5, 6, 7, 8, 9}, dc=range(10, 16), title="Example 08-03.4 with don't cares")


def neighbors(m, n=4):
    return [m ^ (1 << k) for k in range(n)]


print("logical neighbors of m0:", neighbors(0), "-> m8 wraps top-bottom, m2 wraps left-right")
assert set(neighbors(0)) == {1, 2, 4, 8}

logical neighbors of m0: [1, 2, 4, 8] -> m8 wraps top-bottom, m2 wraps left-right


## Part 2 · Quine–McCluskey and Petrick

In [3]:
def quine_mccluskey(on, dc, n, verbose=False):
    # Return the prime implicants (as strings) of the function with on-set `on` and don't cares `dc`.
    terms = {to_bits(i, n) for i in set(on) | set(dc)}
    primes = set()
    stage = 0
    while terms:
        groups = {}
        for t in terms:
            groups.setdefault(t.count("1"), set()).add(t)
        merged, used = set(), set()
        for k in sorted(groups):
            for a in groups[k]:
                for b in groups.get(k + 1, ()):
                    diff = [i for i in range(n) if a[i] != b[i]]
                    if len(diff) == 1 and "-" not in (a[diff[0]], b[diff[0]]):
                        merged.add(a[:diff[0]] + "-" + a[diff[0] + 1:])
                        used |= {a, b}
        primes |= terms - used
        if verbose:
            print(f"stage {stage}: {len(terms)} terms, {len(merged)} merged, primes so far {sorted(primes)}")
        terms, stage = merged, stage + 1
    return sorted(primes)


def petrick(on, primes, n):
    # Product of sums of primes covering each minterm, multiplied out with absorption.
    products = [frozenset()]
    for m in sorted(on):
        choices = [p for p in primes if covers(p, m, n)]
        products = {prod | {p} for prod in products for p in choices}
        products = [p for p in products if not any(q < p for q in products)]   # X + XY = X
    cost = lambda s: (len(s), sum(literals(t) for t in s))
    best = min(cost(s) for s in products)
    return [sorted(s) for s in products if cost(s) == best], best


on6 = [0, 4, 5, 7, 8, 9, 13, 15]
P6 = quine_mccluskey(on6, [], 4, verbose=True)
print("primes:", [term_str(p) for p in P6])
ess = [p for p in P6 if any(sum(covers(q, m, 4) for q in P6) == 1 for m in on6 if covers(p, m, 4))]
print("essential:", [term_str(p) for p in ess])
sols, cost = petrick(on6, P6, 4)
print("minimal covers:", [" + ".join(term_str(t) for t in s) for s in sols], "cost (terms, literals) =", cost)
assert len(P6) == 6 and [term_str(p) for p in ess] == ["BD"] and cost == (3, 8) and len(sols) == 1

on13 = [0, 1, 5, 7, 8, 10, 14, 15]
P13 = quine_mccluskey(on13, [], 4)
sols13, cost13 = petrick(on13, P13, 4)
print("\ncyclic function: primes", len(P13), "-> minimal covers:")
for s in sols13:
    print("  ", " + ".join(term_str(t) for t in s))
assert len(P13) == 8 and len(sols13) == 2 and cost13 == (4, 12)

stage 0: 8 terms, 9 merged, primes so far []
stage 1: 9 terms, 1 merged, primes so far ['-000', '0-00', '010-', '1-01', '100-']
stage 2: 1 terms, 0 merged, primes so far ['-000', '-1-1', '0-00', '010-', '1-01', '100-']
primes: ["B'C'D'", 'BD', "A'C'D'", "A'BC'", "AC'D", "AB'C'"]
essential: ['BD']
minimal covers: ["BD + A'C'D' + AB'C'"] cost (terms, literals) = (3, 8)

cyclic function: primes 8 -> minimal covers:
   A'B'C' + A'BD + AB'D' + ABC
   B'C'D' + BCD + A'C'D + ACD'


## Part 3 · An exact referee, and how the effort grows

An exact minimum cover by dynamic programming over subsets of minterms. It needs the primes (from QM) and is exact, but its effort explodes with $n$.

**🔮 Predict:** if a random 4-input function takes a few milliseconds, how long does a random 6-input function take?

In [4]:
def exact_cover(on, primes, n):
    # minimum (terms, literals) over all covers, by recursion on the lowest uncovered minterm
    idx = {m: k for k, m in enumerate(sorted(on))}
    pm = [(sum(1 << idx[m] for m in on if covers(p, m, n)), literals(p)) for p in primes]

    @functools.lru_cache(None)
    def best(rem):
        if rem == 0:
            return (0, 0)
        low = rem & -rem
        options = []
        for m, cost in pm:
            if m & low:
                terms, lits = best(rem & ~m)
                options.append((terms + 1, lits + cost))
        return min(options)
    return best((1 << len(on)) - 1)


rng = np.random.default_rng(803)
for n in (3, 4, 5, 6):
    times, nprimes = [], []
    for _ in range(4):
        on = [i for i in range(2 ** n) if rng.random() < 0.5] or [0]
        t0 = time.perf_counter()
        P = quine_mccluskey(on, [], n)
        exact_cover(on, P, n)
        times.append(time.perf_counter() - t0)
        nprimes.append(len(P))
    print(f"n = {n}: mean primes {np.mean(nprimes):5.1f}, mean time {1e3 * np.mean(times):8.2f} ms")

# the referee agrees with Petrick on the chapter's example
assert exact_cover(on6, P6, 4) == (3, 8)

n = 3: mean primes   2.8, mean time     0.06 ms
n = 4: mean primes   6.0, mean time     0.17 ms
n = 5: mean primes  11.5, mean time     0.53 ms
n = 6: mean primes  26.5, mean time     2.61 ms


## Part 4 · What don't cares buy (the chapter's guided exercise)

Every three-variable function in which each row is 1, 0 or don't care: $3^8$ = 6561 functions. For each, find the minimum literal count twice, with don't cares as 0s and with them free.

**🔮 Predict:** by what percentage do the don't cares cut the average?

In [5]:
def min_literals(on, dc, n):
    if not on:
        return 0
    P = quine_mccluskey(on, dc, n)
    idx = {m: k for k, m in enumerate(sorted(on))}
    pm = [(sum(1 << idx[m] for m in on if covers(p, m, n)), literals(p)) for p in P]

    @functools.lru_cache(None)
    def best(rem):
        if rem == 0:
            return 0
        low = rem & -rem
        return min(best(rem & ~m) + lit for m, lit in pm if m & low)
    return best((1 << len(on)) - 1)


tot0 = tot1 = 0
for vals in itertools.product((0, 1, 2), repeat=8):
    on = [i for i, x in enumerate(vals) if x == 1]
    dc = [i for i, x in enumerate(vals) if x == 2]
    tot0 += min_literals(on, [], 3)
    tot1 += min_literals(on, dc, 3)
N = 3 ** 8
print(f"{N} functions: mean literals {tot0 / N:.3f} without don't cares, {tot1 / N:.3f} with them,"
      f" a {100 * (1 - tot1 / tot0):.1f}% cut")
assert N == 6561 and round(tot0 / N, 3) == 4.360 and round(tot1 / N, 3) == 2.555

6561 functions: mean literals 4.360 without don't cares, 2.555 with them, a 41.4% cut


About a 41% cut: don't cares let groups grow, and each doubling deletes a literal. The price is that the circuit's outputs on those inputs are no longer chosen by anyone (see the Verilog testbench).

## What to take away

1. Gray-coded maps make one-bit neighbors visible, including across the edges.
2. Minimization is prime generation followed by covering; essentials are forced, the rest is a choice.
3. Exact covering blows up quickly, which is why tools are heuristic.
4. Don't cares are powerful and silently define behavior on "impossible" inputs.